# Aula 05 — Busca informada: Greedy e A*

Este notebook complementa a Aula 05 de **Inteligência Artificial Aplicada**.

## Objetivos

Ao final, você deverá ser capaz de:

- representar um problema de rotas como um grafo com custos;
- construir uma heurística a partir da distância em linha reta até o objetivo;
- executar e interpretar a **Busca Greedy (Greedy Best-First Search)**;
- executar e interpretar o **A\***;
- comparar **h(n)**, **g(n)** e **f(n) = g(n) + h(n)**.

O notebook contém:

- **1 exemplo resolvido de Busca Greedy**;
- **2 exercícios de Busca Greedy**;
- **1 exemplo resolvido de A\***;
- **2 exercícios de A\***.

Os mapas foram propositalmente construídos com várias alternativas. A ideia é evitar problemas em que o melhor caminho seja evidente apenas olhando a figura.

## 1. Ferramentas que usaremos

Cada cenário será descrito por:

- `conexoes`: trechos possíveis no formato `(origem, destino, custo)`;
- `posicoes`: coordenadas usadas para desenhar o mapa;
- `mapa`: lista de sucessores de cada estado;
- `heuristica`: estimativa em linha reta até o objetivo.

A heurística será calculada pela distância euclidiana entre cada ponto e o objetivo. Como os custos dos trechos são maiores ou iguais à distância geométrica entre os pontos, essa heurística funciona como uma estimativa otimista do custo restante.

In [ ]:
from heapq import heappush, heappop
from itertools import count
import math

import matplotlib.pyplot as plt
import networkx as nx


def criar_mapa(conexoes):
    """Converte uma lista (origem, destino, custo) em um grafo não direcionado."""
    mapa = {}
    for origem, destino, custo in conexoes:
        mapa.setdefault(origem, []).append((destino, custo))
        mapa.setdefault(destino, []).append((origem, custo))
    return mapa


def criar_heuristica(posicoes, objetivo):
    """Distância em linha reta até o objetivo, na mesma escala dos custos."""
    x_obj, y_obj = posicoes[objetivo]
    return {
        estado: math.floor(math.dist((x, y), (x_obj, y_obj)) * 10)
        for estado, (x, y) in posicoes.items()
    }


def mostrar_mapa(conexoes, posicoes, heuristica, inicio, objetivo, titulo):
    """Desenha o grafo mostrando custo nas arestas e h(n) nos estados."""
    grafo = nx.Graph()
    for origem, destino, custo in conexoes:
        grafo.add_edge(origem, destino, weight=custo)

    labels_nos = {
        estado: f"{estado}\nh={heuristica[estado]}"
        for estado in grafo.nodes
    }
    labels_arestas = nx.get_edge_attributes(grafo, "weight")

    plt.figure(figsize=(13, 7))
    nx.draw(
        grafo,
        posicoes,
        labels=labels_nos,
        with_labels=True,
        node_size=2300,
        font_size=8,
        width=1.4,
    )
    nx.draw_networkx_edge_labels(
        grafo,
        posicoes,
        edge_labels=labels_arestas,
        font_size=8,
    )
    plt.title(f"{titulo}\nInício: {inicio}  |  Objetivo: {objetivo}")
    plt.axis("off")
    plt.show()


## 2. Algoritmos

Observe que os dois algoritmos usam uma fila de prioridade.

A diferença central é o valor usado como prioridade:

- **Greedy:** `h(n)`
- **A\*:** `g(n) + h(n)`

In [ ]:
def busca_greedy(estado_inicial, objetivo, mapa, heuristica, mostrar_passos=False):
    fronteira = []
    ordem = count()

    heappush(
        fronteira,
        (heuristica[estado_inicial], next(ordem), estado_inicial, [estado_inicial], 0)
    )

    visitados = set()
    expandidos = []

    while fronteira:
        prioridade, _, estado, caminho, custo = heappop(fronteira)

        if estado in visitados:
            continue

        visitados.add(estado)
        expandidos.append(estado)

        if mostrar_passos:
            print(
                f"Retirado: {estado:12} | h={prioridade:3} | "
                f"g={custo:3} | caminho={' -> '.join(caminho)}"
            )

        if estado == objetivo:
            return {
                "caminho": caminho,
                "custo": custo,
                "expandidos": expandidos,
            }

        for sucessor, custo_acao in mapa[estado]:
            if sucessor not in visitados:
                heappush(
                    fronteira,
                    (
                        heuristica[sucessor],
                        next(ordem),
                        sucessor,
                        caminho + [sucessor],
                        custo + custo_acao,
                    ),
                )

    return None


def a_estrela(estado_inicial, objetivo, mapa, heuristica, mostrar_passos=False):
    fronteira = []
    ordem = count()

    melhor_g = {estado_inicial: 0}

    heappush(
        fronteira,
        (heuristica[estado_inicial], next(ordem), estado_inicial, [estado_inicial], 0)
    )

    expandidos = []

    while fronteira:
        prioridade, _, estado, caminho, g_atual = heappop(fronteira)

        if g_atual > melhor_g[estado]:
            continue

        expandidos.append(estado)

        if mostrar_passos:
            print(
                f"Retirado: {estado:12} | g={g_atual:3} | "
                f"h={heuristica[estado]:3} | f={prioridade:3}"
            )

        if estado == objetivo:
            return {
                "caminho": caminho,
                "custo": g_atual,
                "expandidos": expandidos,
            }

        for sucessor, custo_acao in mapa[estado]:
            novo_g = g_atual + custo_acao

            if sucessor not in melhor_g or novo_g < melhor_g[sucessor]:
                melhor_g[sucessor] = novo_g

                heappush(
                    fronteira,
                    (
                        novo_g + heuristica[sucessor],
                        next(ordem),
                        sucessor,
                        caminho + [sucessor],
                        novo_g,
                    ),
                )

    return None


def imprimir_resultado(resultado):
    print("Caminho:", " -> ".join(resultado["caminho"]))
    print("Custo:", resultado["custo"])
    print("Estados expandidos:", " -> ".join(resultado["expandidos"]))


# Parte A — Busca Greedy

## Exemplo resolvido — Robô de coleta em um centro logístico

Um robô parte da **Entrada** e precisa chegar à **Expedição**. Os custos representam tempo estimado de deslocamento entre setores.

A heurística `h(n)` é calculada pela distância em linha reta até a Expedição.

Antes de executar o algoritmo, observe o mapa e tente prever qual sequência a Greedy escolherá. Lembre-se: ela considera **apenas h(n)** para decidir qual alternativa retirar da fronteira.

In [ ]:
inicio_g0 = "Entrada"
objetivo_g0 = "Expedicao"

pos_g0 = {
    "Entrada": (0, 3.0),
    "Separacao": (6.1, 4.8),
    "Estoque_A": (2.7, 7.9),
    "Estoque_B": (7.5, 5.1),
    "Conferencia": (5.0, 2.4),
    "Triagem": (1.0, 3.7),
    "Docas": (5.7, 0.7),
    "Embalagem": (7.4, 2.3),
    "Retorno": (2.9, 0.4),
    "Pre_Exp": (9.0, 7.4),
    "Expedicao": (10, 6.1),
}

conexoes_g0 = [
    ("Entrada", "Triagem", 23),
    ("Separacao", "Estoque_B", 16),
    ("Separacao", "Conferencia", 31),
    ("Estoque_A", "Triagem", 131),
    ("Estoque_B", "Pre_Exp", 107),
    ("Conferencia", "Docas", 29),
    ("Conferencia", "Embalagem", 27),
    ("Triagem", "Retorno", 69),
    ("Docas", "Retorno", 52),
    ("Pre_Exp", "Expedicao", 34),
    ("Entrada", "Separacao", 72),
    ("Triagem", "Pre_Exp", 149),
    ("Triagem", "Embalagem", 229),
    ("Separacao", "Embalagem", 39),
    ("Entrada", "Estoque_A", 92),
    ("Docas", "Pre_Exp", 120),
    ("Docas", "Expedicao", 372),
    ("Separacao", "Pre_Exp", 160),
]

mapa_g0 = criar_mapa(conexoes_g0)
heuristica_g0 = criar_heuristica(pos_g0, objetivo_g0)

mostrar_mapa(
    conexoes_g0, pos_g0, heuristica_g0,
    inicio_g0, objetivo_g0,
    "Exemplo Greedy — Centro logístico"
)


In [ ]:
resultado_g0 = busca_greedy(
    inicio_g0,
    objetivo_g0,
    mapa_g0,
    heuristica_g0,
    mostrar_passos=True,
)

print()
imprimir_resultado(resultado_g0)


### O que observar no exemplo

1. A Greedy escolhe sempre o menor `h`, mesmo que o caminho percorrido já tenha acumulado um custo alto.
2. Um estado próximo geometricamente do objetivo pode ser alcançado por uma conexão cara.
3. Encontrar rapidamente o objetivo **não significa** encontrar a rota de menor custo.

Experimente depois executar o A\* nesse mesmo mapa e compare os resultados.

## Exercício Greedy 1 — Inspeção em uma planta industrial

Um técnico precisa sair da **Portaria** e chegar à **Subestação**. O custo de cada conexão representa o tempo necessário para atravessar aquele trecho.

### Tarefa

1. Execute a célula do mapa.
2. Antes de rodar a Greedy, anote qual rota você acredita que será escolhida.
3. Complete a célula com `TODO`.
4. Registre:
   - caminho retornado;
   - custo total;
   - ordem dos estados expandidos.
5. Explique em uma frase por que a Greedy tomou essa decisão.

In [ ]:
inicio_g1 = "Portaria"
objetivo_g1 = "Subestacao"

pos_g1 = {
    "Portaria": (0, 5.9),
    "Caldeira": (8.7, 2.0),
    "Utilidades": (1.5, 8.2),
    "Producao": (3.7, 3.7),
    "Bombas": (7.5, 1.8),
    "Painel": (8.3, 4.9),
    "Oficina": (1.6, 9.2),
    "Tanques": (5.6, 0.8),
    "Compressores": (7.7, 7.7),
    "Transformador": (6.9, 6.6),
    "Subestacao": (10, 6.2),
}

conexoes_g1 = [
    ("Portaria", "Utilidades", 53),
    ("Portaria", "Producao", 50),
    ("Caldeira", "Bombas", 16),
    ("Caldeira", "Painel", 64),
    ("Utilidades", "Oficina", 42),
    ("Producao", "Tanques", 48),
    ("Bombas", "Tanques", 28),
    ("Painel", "Transformador", 23),
    ("Painel", "Subestacao", 32),
    ("Compressores", "Transformador", 22),
    ("Oficina", "Transformador", 93),
    ("Compressores", "Subestacao", 41),
    ("Caldeira", "Utilidades", 413),
    ("Portaria", "Tanques", 125),
    ("Utilidades", "Transformador", 71),
    ("Caldeira", "Subestacao", 159),
    ("Tanques", "Transformador", 67),
    ("Bombas", "Subestacao", 60),
]

heuristica_g1 = criar_heuristica(pos_g1, objetivo_g1)

mostrar_mapa(
    conexoes_g1, pos_g1, heuristica_g1,
    inicio_g1, objetivo_g1,
    "Exercício Greedy 1 — Planta industrial"
)


In [ ]:
# TODO — complete as três linhas.

mapa_g1 = ...

resultado_g1 = ...

# Depois de executar a busca:
# imprimir_resultado(resultado_g1)


**Resposta curta do estudante**

- Minha previsão antes de executar:
- Caminho encontrado:
- Custo:
- Por que a Greedy escolheu esse caminho?


## Exercício Greedy 2 — Veículo autônomo em um campus

Um veículo interno parte da **Garagem** e precisa chegar ao **Laboratório**. Há várias vias internas, algumas curtas geometricamente, mas caras em tempo de deslocamento.

### Tarefa

Monte o `mapa`, execute a Greedy e responda:

1. Qual estado é retirado primeiro depois da Garagem?
2. Qual caminho é devolvido?
3. A menor heurística em cada passo foi suficiente para produzir uma rota barata?
4. Identifique uma decisão em que `h(n)` parece atraente, mas o custo acumulado já deveria causar desconfiança.

In [ ]:
inicio_g2 = "Garagem"
objetivo_g2 = "Laboratorio"

pos_g2 = {
    "Garagem": (0, 5.3),
    "Biblioteca": (7.8, 2.9),
    "Bloco_C": (5.1, 3.4),
    "Ginasio": (4.3, 9.7),
    "Cantina": (1.8, 4.4),
    "Auditorio": (2.8, 3.5),
    "Estacionamento": (9.0, 3.3),
    "Bloco_A": (5.9, 4.2),
    "Bloco_B": (6.8, 4.5),
    "Praca": (2.4, 4.3),
    "Laboratorio": (10, 4.1),
}

conexoes_g2 = [
    ("Garagem", "Cantina", 34),
    ("Biblioteca", "Estacionamento", 20),
    ("Biblioteca", "Bloco_B", 31),
    ("Bloco_C", "Bloco_A", 21),
    ("Bloco_C", "Auditorio", 41),
    ("Ginasio", "Praca", 102),
    ("Cantina", "Praca", 7),
    ("Auditorio", "Praca", 28),
    ("Estacionamento", "Laboratorio", 42),
    ("Bloco_A", "Bloco_B", 16),
    ("Ginasio", "Cantina", 93),
    ("Cantina", "Estacionamento", 132),
    ("Estacionamento", "Praca", 138),
    ("Bloco_C", "Cantina", 74),
    ("Garagem", "Bloco_B", 149),
    ("Garagem", "Bloco_A", 82),
    ("Bloco_C", "Laboratorio", 54),
    ("Auditorio", "Bloco_B", 173),
]

heuristica_g2 = criar_heuristica(pos_g2, objetivo_g2)

mostrar_mapa(
    conexoes_g2, pos_g2, heuristica_g2,
    inicio_g2, objetivo_g2,
    "Exercício Greedy 2 — Campus"
)


In [ ]:
# TODO — monte o mapa e execute a Greedy.

mapa_g2 = ...

resultado_g2 = ...

# imprimir_resultado(resultado_g2)


# Parte B — Algoritmo A*

## Exemplo resolvido — Rota de atendimento de emergência

Uma equipe sai da **Base** e precisa chegar ao **Hospital**. O custo de cada trecho representa tempo de deslocamento.

Aqui a prioridade não é apenas `h(n)`:

\[
f(n) = g(n) + h(n)
\]

- `g(n)`: custo realmente acumulado desde a Base;
- `h(n)`: estimativa do que ainda falta;
- `f(n)`: estimativa do custo total da rota que passa por aquele estado.

In [ ]:
inicio_a0 = "Base"
objetivo_a0 = "Hospital"

pos_a0 = {
    "Base": (0, 6.3),
    "Bairro_N": (2.5, 10.0),
    "Avenida_1": (2.6, 6.7),
    "Avenida_2": (1.7, 7.6),
    "Praca": (2.2, 7.1),
    "Viaduto": (6.8, 7.7),
    "Terminal": (4.5, 8.7),
    "Tunel": (8.9, 1.2),
    "Contorno": (5.1, 9.6),
    "Centro": (6.9, 4.4),
    "Hospital": (10, 7.0),
}

conexoes_a0 = [
    ("Base", "Avenida_2", 25),
    ("Bairro_N", "Terminal", 92),
    ("Bairro_N", "Avenida_2", 37),
    ("Avenida_1", "Praca", 7),
    ("Avenida_2", "Praca", 35),
    ("Viaduto", "Terminal", 40),
    ("Viaduto", "Centro", 55),
    ("Viaduto", "Hospital", 42),
    ("Terminal", "Contorno", 22),
    ("Tunel", "Centro", 55),
    ("Bairro_N", "Viaduto", 86),
    ("Bairro_N", "Centro", 114),
    ("Avenida_1", "Terminal", 46),
    ("Praca", "Viaduto", 179),
    ("Praca", "Tunel", 126),
    ("Praca", "Contorno", 51),
    ("Avenida_2", "Viaduto", 159),
    ("Base", "Tunel", 147),
]

mapa_a0 = criar_mapa(conexoes_a0)
heuristica_a0 = criar_heuristica(pos_a0, objetivo_a0)

mostrar_mapa(
    conexoes_a0, pos_a0, heuristica_a0,
    inicio_a0, objetivo_a0,
    "Exemplo A* — Atendimento de emergência"
)


In [ ]:
resultado_a0 = a_estrela(
    inicio_a0,
    objetivo_a0,
    mapa_a0,
    heuristica_a0,
    mostrar_passos=True,
)

print()
imprimir_resultado(resultado_a0)


### O que observar no exemplo

O A\* pode deixar de escolher o estado que tem o menor `h` isoladamente.

Isso ocorre porque uma alternativa só é realmente interessante quando consideramos:

**quanto já custou chegar até ela + quanto estimamos que ainda falta.**

Esse é o papel de `f(n) = g(n) + h(n)`.

## Exercício A* 1 — Drone de inspeção

Um drone parte do **Hangar** e deve alcançar a **Torre**. O custo das conexões considera distância, consumo de bateria e restrições de voo.

### Tarefa

1. Monte o mapa.
2. Gere a heurística.
3. Execute A\* mostrando os passos.
4. Para os três primeiros estados retirados, registre `g`, `h` e `f`.
5. Explique por que o menor `h` sozinho não seria um critério suficiente.

In [ ]:
inicio_a1 = "Hangar"
objetivo_a1 = "Torre"

pos_a1 = {
    "Hangar": (0, 3.9),
    "Reservatorio": (6.2, 0.7),
    "Galpao": (5.3, 3.7),
    "Chamine": (1.5, 5.1),
    "Patio": (1.3, 4.3),
    "Oficina": (1.6, 0.9),
    "Silos": (4.4, 8.3),
    "Linha_1": (2.0, 2.2),
    "Linha_2": (6.0, 9.5),
    "Antena": (5.6, 4.0),
    "Torre": (10, 2.9),
}

conexoes_a1 = [
    ("Hangar", "Patio", 17),
    ("Reservatorio", "Galpao", 62),
    ("Reservatorio", "Torre", 76),
    ("Galpao", "Antena", 7),
    ("Galpao", "Linha_1", 83),
    ("Chamine", "Patio", 14),
    ("Chamine", "Silos", 62),
    ("Patio", "Linha_1", 35),
    ("Oficina", "Linha_1", 28),
    ("Silos", "Linha_2", 27),
    ("Antena", "Torre", 75),
    ("Chamine", "Linha_1", 55),
    ("Oficina", "Silos", 396),
    ("Galpao", "Chamine", 187),
    ("Galpao", "Torre", 53),
    ("Hangar", "Silos", 119),
    ("Chamine", "Oficina", 86),
    ("Reservatorio", "Silos", 144),
]

heuristica_a1 = criar_heuristica(pos_a1, objetivo_a1)

mostrar_mapa(
    conexoes_a1, pos_a1, heuristica_a1,
    inicio_a1, objetivo_a1,
    "Exercício A* 1 — Drone de inspeção"
)


In [ ]:
# TODO — complete e execute.

mapa_a1 = ...

resultado_a1 = ...

# imprimir_resultado(resultado_a1)


## Exercício A* 2 — Robô de manutenção em um data center

Um robô de manutenção parte do **NOC** e precisa chegar à **Sala_Critica**. O custo representa tempo de circulação entre corredores, portas de segurança e zonas de acesso.

### Tarefa

1. Monte `mapa_a2`.
2. Gere `heuristica_a2`.
3. Execute A\* com `mostrar_passos=True`.
4. Identifique um momento em que um estado com `h` menor não é escolhido porque seu `g` tornou `f` pior.
5. Responda: por que a heurística em linha reta é admissível neste modelo?

In [ ]:
inicio_a2 = "NOC"
objetivo_a2 = "Sala_Critica"

pos_a2 = {
    "NOC": (0, 5.1),
    "Corredor_A": (8.7, 2.9),
    "Corredor_B": (7.1, 7.0),
    "UPS": (6.3, 1.1),
    "Recepcao": (1.2, 3.8),
    "Rack_1": (7.0, 2.5),
    "Rack_2": (5.0, 3.2),
    "Climatizacao": (7.8, 9.5),
    "Seguranca": (4.2, 10.0),
    "Geradores": (1.5, 8.1),
    "Sala_Critica": (10, 6.8),
}

conexoes_a2 = [
    ("NOC", "Recepcao", 37),
    ("NOC", "Geradores", 138),
    ("Corredor_A", "Rack_1", 28),
    ("Corredor_B", "Climatizacao", 31),
    ("Corredor_B", "Sala_Critica", 58),
    ("UPS", "Rack_1", 17),
    ("Recepcao", "Rack_2", 43),
    ("Rack_1", "Rack_2", 31),
    ("Climatizacao", "Seguranca", 54),
    ("Seguranca", "Geradores", 68),
    ("Corredor_B", "Geradores", 97),
    ("Corredor_B", "Rack_1", 75),
    ("Corredor_A", "Seguranca", 114),
    ("UPS", "Climatizacao", 108),
    ("UPS", "Recepcao", 98),
    ("Corredor_A", "Climatizacao", 80),
    ("NOC", "Corredor_A", 189),
    ("Corredor_B", "Rack_2", 64),
]

heuristica_a2 = criar_heuristica(pos_a2, objetivo_a2)

mostrar_mapa(
    conexoes_a2, pos_a2, heuristica_a2,
    inicio_a2, objetivo_a2,
    "Exercício A* 2 — Data center"
)


In [ ]:
# TODO — complete e execute.

mapa_a2 = ...

resultado_a2 = ...

# imprimir_resultado(resultado_a2)


# Fechamento

Use os resultados dos seis cenários para completar a comparação:

| Questão | Greedy | A* |
|---|---|---|
| Qual é a prioridade? | `h(n)` | `g(n) + h(n)` |
| O custo já percorrido influencia a escolha? | Não | Sim |
| Uma heurística admissível garante rota ótima? | Não | Nas condições discutidas na aula, sim |
| Pode chegar rapidamente a uma solução cara? | Sim | A heurística admissível evita encerrar em uma solução mais cara nas condições do A* estudado |

## Questão final

Explique com suas palavras:

> Por que uma boa estimativa ajuda uma busca, mas somente o A\* combina essa estimativa com o custo que já foi realmente pago?
